# Runbook 1.2.b: Weak authentication patterns; adversary dictionary construction

**Skill:** Analyze Risk · **Maps to:** 1.2.B · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Build the password-weakness checker an authentication backend would use to give users actionable
feedback when they choose a new password. Given a password string, return a report dict
identifying whether the password is weak and which specific rules it violates.

**Lesson objective.** Evaluate a password against the core weakness criteria (length, dictionary membership, character variety, and sequential/repeated patterns) and explain how adversaries build and use wordlists to attack weak passwords.

### What comes in
A single string, the candidate password. May be empty.

### What must come out
A dict with exactly two keys:
- `"weak"` (`bool`), `True` if the password fails ANY rule; `False` only if it passes all rules.
- `"reasons"` (`list[str]`), sorted list of rule-labels for each rule the password fails.
  Empty list if the password passes all rules.

Possible reason labels (exact strings, lowercase):
- `"too_short"`: length < 12 characters
- `"in_dictionary"`: password (lowercased) is in the built-in common wordlist
- `"only_lowercase"`: contains no uppercase letters, digits, or non-alphabetic characters
- `"sequential_or_repeated"`: contains a sequential run of 3+ characters (e.g. `abc`, `123`,
  `xyz`) OR a repeated run of 3+ identical characters (e.g. `aaa`, `111`)

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
solve("password")
-> {"weak": True, "reasons": ["in_dictionary", "only_lowercase", "too_short"]}

solve("P@ssword!2024")
-> {"weak": True, "reasons": ["in_dictionary"]}

solve("Tr0ub4dor&3abcde")
-> {"weak": True, "reasons": ["sequential_or_repeated"]}

solve("mXk9#vLq2@nP")
-> {"weak": False, "reasons": []}
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] The common wordlist is defined in the Reference section below and must be embedded in your solution.
- [ ] Reasons list is always sorted ascending.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(password: str) -> dict
```

#### Input (`password`)
A single `str`, the candidate password. May be empty or contain any Unicode.

#### Output (`result`)
A `dict` with exactly these keys:
- `"weak"` (`bool`), `True` if any rule below is triggered; `False` if all rules pass.
- `"reasons"` (`list[str]`), sorted list of triggered rule labels (see below). Empty if no rules triggered.

#### Rules (all must be checked; collect all failures)

##### Rule 1: `"too_short"`
Triggered when `len(password) < 12`.

##### Rule 2: `"in_dictionary"`
Triggered when `password.lower()` is a member of the COMMON_WORDS set below.
```
COMMON_WORDS = {
    "password", "password1", "password123", "123456", "123456789", "12345678",
    "qwerty", "qwerty123", "letmein", "iloveyou", "admin", "welcome",
    "monkey", "dragon", "master", "abc123", "sunshine", "princess",
    "shadow", "superman", "football", "baseball", "login", "passw0rd",
    "pa$$word", "access", "mustang", "1234567890", "starwars", "hello",
}
```

##### Rule 3: `"only_lowercase"`
Triggered when the password contains ONLY lowercase ASCII letters (`a`-`z`), i.e.,
`password.isalpha() and password == password.lower()`.
Note: an empty string does NOT trigger this rule (it has no lowercase letters either).

##### Rule 4: `"sequential_or_repeated"`
Triggered when the password contains:
- A run of 3 or more consecutive ASCII characters in ascending order
  (e.g. `abc`, `bcd`, `xyz`, `123`, `234`, `789`), OR
- A run of 3 or more identical characters (e.g. `aaa`, `111`, `   `)

Detection algorithm:
For each window of 3 consecutive characters at position i:
- Repeated: `c[i] == c[i+1] == c[i+2]`
- Sequential ascending: `ord(c[i+1]) - ord(c[i]) == 1` AND `ord(c[i+2]) - ord(c[i+1]) == 1`

If any window matches either condition, the rule is triggered.

#### Output contract
- `"reasons"` is always sorted ascending (alphabetical order of the label strings).
- `"weak"` equals `len(reasons) > 0`.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- The COMMON_WORDS set above is the exact set used by tests.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

Fill in `solve` below. Run the cell (Shift+Enter) after each change.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
"""Starter code for 1.2.b. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""


def solve(password: str) -> dict:
    """Return a weakness report dict for the given password.

    Args:
        password: the candidate password string (may be empty).
    Returns:
        {"weak": bool, "reasons": list[str]} where reasons is a sorted list of
        triggered rule labels. Possible labels:
          "too_short"            -- len(password) < 12
          "in_dictionary"        -- password.lower() is in the common wordlist
          "only_lowercase"       -- password contains only a-z characters
          "sequential_or_repeated" -- run of 3+ sequential or identical chars
        See the Reference section below for the exact COMMON_WORDS set and detection algorithm.
    """
    # TODO: implement all four rules and return {"weak": ..., "reasons": [...]}
    raise NotImplementedError("implement solve() -- see the problem statement above")


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_dict_only_lc_short():
    result = impl.solve("password")
    assert result["weak"] is True
    assert result["reasons"] == ["in_dictionary", "only_lowercase", "too_short"]


def test_sequential_only():
    result = impl.solve("Tr0ub4dor&3abcde")
    assert result["weak"] is True
    assert result["reasons"] == ["sequential_or_repeated"]


def test_strong():
    result = impl.solve("mXk9#vLq2@nP")
    assert result["weak"] is False
    assert result["reasons"] == []


def test_dictionary_with_mangling():
    result = impl.solve("pa$$word")
    assert result["weak"] is True
    assert "in_dictionary" in result["reasons"]
    assert "too_short" in result["reasons"]


def test_empty_string():
    result = impl.solve("")
    assert result["weak"] is True
    assert result["reasons"] == ["too_short"]


def test_repeated_chars():
    result = impl.solve("HelloWorld111")
    assert result["weak"] is True
    assert result["reasons"] == ["sequential_or_repeated"]


def test_only_lowercase_flag():
    result = impl.solve("sunshine")
    assert "only_lowercase" in result["reasons"]
    assert "too_short" in result["reasons"]
    assert "in_dictionary" in result["reasons"]


def test_reasons_sorted():
    result = impl.solve("qwerty")
    assert result["reasons"] == sorted(result["reasons"])


def test_sequential_digits():
    result = impl.solve("Abcdef123456!")
    assert result["weak"] is True
    assert "sequential_or_repeated" in result["reasons"]


def test_exactly_12_chars_passes_length():
    result = impl.solve("zQm7!kLp3$Rw")
    assert "too_short" not in result["reasons"]


def test_weak_flag_matches_reasons():
    for pw in ["mXk9#vLq2@nP", "password", "HelloWorld111", ""]:
        result = impl.solve(pw)
        assert result["weak"] == (len(result["reasons"]) > 0)


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve("password"),
     {"weak": True, "reasons": ["in_dictionary", "only_lowercase", "too_short"]}),
    (lambda: solve("Tr0ub4dor&3abcde"),
     {"weak": True, "reasons": ["sequential_or_repeated"]}),
    (lambda: solve("mXk9#vLq2@nP"),
     {"weak": False, "reasons": []}),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it